# 🐚 MiniMax-H3（海螺 3 开源版 · 33B 音视频）ComfyUI 专用引擎

- **硬件要求**：强烈建议选择 **A100（40GB/80GB）+ 高 RAM**；**L4（24GB + 高 RAM）** 需配合 Q4/GGUF 量化勉强运行；**T4 (16GB) 跑不动**。
- **省单元设计**：所有模型权重和输出视频都会直接保存在你的 Google Drive 中，避免重复下载 20–30GB 权重浪费计算单元。
- **用完务必运行第 4 步**释放算力！


In [ ]:
#@title 1. 检查硬件 + 挂载 Google Drive + 安装 ComfyUI 与 MiniMax-H3 支持
import os, subprocess, torch, psutil

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0
ram_gb = psutil.virtual_memory().total / (1024 ** 3)

print(f"🖥️  当前 GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) | 系统内存: {ram_gb:.1f} GB RAM")
if vram_gb < 20:
    print("⚠️ 警告：MiniMax-H3 是 33B 全模态模型，T4 (16GB) 极大概率显存/内存溢出！")
    print("👉 强烈建议切换到 A100 (40GB/80GB) 或 L4 (24GB + 高 RAM)，或改用主引擎跑 Wan 2.2 / LTX-Video。")

from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODELS = "/content/drive/MyDrive/ColabMediaStudio/ComfyUI_MiniMax_H3_Models"
DRIVE_OUTPUTS = "/content/drive/MyDrive/ColabMediaStudio/MiniMax_H3_Outputs"
os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

if not os.path.exists("/content/ComfyUI"):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", "/content/ComfyUI"])

subprocess.check_call(["pip", "install", "-q", "-U", "-r", "/content/ComfyUI/requirements.txt", "huggingface_hub", "hf_transfer"])

gguf_dir = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"
if not os.path.exists(gguf_dir):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/city96/ComfyUI-GGUF.git", gguf_dir])
    subprocess.check_call(["pip", "install", "-q", "-U", "-r", f"{gguf_dir}/requirements.txt"])

# Link ComfyUI output & extra model dirs to Google Drive so downloads are cached across sessions
os.makedirs("/content/ComfyUI/models/unet", exist_ok=True)
extra_yaml = f"""
colab_drive:
  base_path: {DRIVE_MODELS}
  checkpoints: checkpoints/
  diffusion_models: diffusion_models/
  unet: unet/
  vae: vae/
  text_encoders: text_encoders/
  clip: clip/
"""
with open("/content/ComfyUI/extra_model_paths.yaml", "w") as f:
    f.write(extra_yaml.strip() + "\n")

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

print("✅ ComfyUI + ComfyUI-GGUF + Google Drive 缓存配置完成！")


In [ ]:
#@title 2. 查看/下载 MiniMax-H3 官方 Comfy-Org 权重到 Google Drive 缓存
import os
from huggingface_hub import HfApi, hf_hub_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DRIVE_MODELS = "/content/drive/MyDrive/ColabMediaStudio/ComfyUI_MiniMax_H3_Models"
REPO_ID = "Comfy-Org/MiniMax-H3" #@param {type:"string"}
AUTO_DOWNLOAD_RECOMMENDED = False #@param {type:"boolean"}

api = HfApi()
try:
    files = api.list_repo_files(REPO_ID)
    print(f"📦 仓库 {REPO_ID} 中可用的文件列表（共 {len(files)} 个）：")
    for fn in files[:40]:
        print("  -", fn)
    if AUTO_DOWNLOAD_RECOMMENDED:
        print("\n⬇️ 正在按目录结构下载所需量化权重到 Google Drive（请确保 Google One 剩余空间 >= 35GB）...")
        for fn in files:
            if fn.endswith((".safetensors", ".gguf")) and ("fp8" in fn.lower() or "vae" in fn.lower() or "text_encoder" in fn.lower()):
                print("下载:", fn)
                hf_hub_download(repo_id=REPO_ID, filename=fn, local_dir=DRIVE_MODELS)
        print("✅ 权重下载并缓存至 Google Drive 完成！")
    else:
        print("\n💡 提示：勾选上方 AUTO_DOWNLOAD_RECOMMENDED=True 可自动拉取 FP8/VAE/文本编码器，")
        print("   或直接在 ComfyUI 网页端的 Templates（模板）里加载 MiniMax-H3 模板按需下载。")
except Exception as e:
    print(f"⚠️ 无法自动列出 {REPO_ID}（可能需要 HuggingFace Token 或仓库路径更新）: {e}")
    print("👉 可手动访问 https://huggingface.co/MiniMaxAI/MiniMax-H3 查看最新权重与 ComfyUI 模板。")


In [ ]:
#@title 3. 启动 ComfyUI 并生成远程访问链接（复制到本地 App 的 MiniMax-H3 专区）
import re, subprocess, time

DRIVE_OUTPUTS = "/content/drive/MyDrive/ColabMediaStudio/MiniMax_H3_Outputs"

comfy_proc = subprocess.Popen(
    [
        "python3", "/content/ComfyUI/main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--output-directory", DRIVE_OUTPUTS,
        "--highvram" if __import__("torch").cuda.get_device_properties(0).total_memory > 60e9 else "--normalvram",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("⏳ 正在启动 ComfyUI（约 15-30 秒）...")
time.sleep(10)

tunnel = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url = None
deadline = time.time() + 45
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

assert url, "❌ 隧道启动失败，请重试。"
print("\n" + "=" * 72)
print("🐚 MiniMax-H3 ComfyUI 工作台已启动！")
print(f"🔗 访问地址（可粘贴进本地 App 的 MiniMax-H3 专区或直接点击）:\n{url}")
print("⚠️ 用完后务必运行下方第 4 步释放 A100/L4 算力，避免持续扣单元！")
print("=" * 72 + "\n")

try:
    for line in comfy_proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    comfy_proc.terminate()
    tunnel.terminate()


In [ ]:
#@title 4. 用完务必运行：立即释放 Colab 运行时，停止扣减计算单元
from google.colab import runtime
print("正在彻底关闭运行时并停止计费...")
runtime.unassign()
